# Literature screening and gene-candidate extraction

Reproducibility notebook for the literature-screening and automated gene-candidate extraction stage of the study on cadmium (Cd), lead (Pb), and mercury (Hg)-associated genes in *Drosophila melanogaster*.

This notebook is a cleaned, linearized version of the original Google Colab workflow used during the study. The analytical rules, term lists, regular expressions, thresholds, and sequence of transformations are preserved from the original notebook; duplicated exploratory cells and Colab-only upload/download commands were removed for portability.

**Input:** `InteractiveSheet_2026-04-23_04_19_46 - Лист1.csv` (507 literature records; literature corpus frozen on 23 April 2026).

**Principal outputs:**
- `screening_semiauto_strict.csv`
- `gene_candidates_auto.csv` (19,095 text-mined candidate mentions after record-level deduplication)
- `validation_1_stage.csv`
- `gene_summary_auto.csv`
- `gene_shortlist_for_manual_validation.csv` (1,053 candidates selected for subsequent manual/identifier validation)

The automated shortlist is a screening output, not a set of validated genes. Subsequent FlyBase identifier validation and evidence-based reassessment are separate stages.


## 1. Setup and input

Place the input CSV in the same working directory as this notebook, or change `INPUT_FILE` below. The notebook uses only `pandas`, `numpy`, and Python's standard `re` module.


In [ ]:
from pathlib import Path
import re
import pandas as pd
import numpy as np

INPUT_FILE = Path("InteractiveSheet_2026-04-23_04_19_46 - Лист1.csv")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

df = pd.read_csv(INPUT_FILE)
print(f"Input records: {len(df):,}")
print(f"Columns: {len(df.columns)}")
assert len(df) == 507, "Unexpected input record count; verify the literature corpus file."
df.head()


## 2. Semi-automatic literature screening

The first pass uses title, abstract, and the original search-query field to classify records as `yes`, `maybe`, or `no`. A second priority pass uses title + abstract only, avoiding inflation of priority by query terms.


In [ ]:
df["screen_text"] = (
    df["title"].fillna("").astype(str) + " " +
    df["abstract"].fillna("").astype(str) + " " +
    df["query"].fillna("").astype(str)
).str.lower()

positive_drosophila = [
    "drosophila",
    "drosophila melanogaster"
]

positive_metals = [
    "cadmium",
    "lead",
    "mercury",
    "heavy metal",
    "metal stress"
]

strong_gene = [
    "gene expression",
    "transcriptome",
    "transcriptomic",
    "toxicogenomic",
    "rna-seq",
    "differential expression",
    "mutant",
    "knockout",
    "rnai",
    "metallothionein",
    "mtf-1",
    "ortholog"
]

moderate_gene = [
    "gene",
    "genes",
    "expression",
    "transcript",
    "pathway",
    "oxidative stress",
    "neuronal",
    "synaptic",
    "apoptosis",
    "mitochondria"
]

negative = [
    "mouse",
    "mice",
    "rat",
    "rats",
    "zebrafish",
    "arabidopsis",
    "plant"
]

def has_any(text, words):
    return any(w in text for w in words)

def classify_strict(text):
    dros = has_any(text, positive_drosophila)
    metal = has_any(text, positive_metals)
    strong = has_any(text, strong_gene)
    moderate = has_any(text, moderate_gene)
    neg = has_any(text, negative)

    if neg and not dros:
        return "no", "not drosophila / likely other model"
    if dros and metal and strong:
        return "yes", "drosophila + metal + strong gene-level evidence"
    if dros and metal and moderate:
        return "maybe", "drosophila + metal + possible molecular relevance"
    return "no", "insufficient relevance"

df[["include_auto", "reason_auto"]] = df["screen_text"].apply(
    lambda x: pd.Series(classify_strict(x))
)

order = {"yes": 0, "maybe": 1, "no": 2}
df["sort_key"] = df["include_auto"].map(order)

df_sorted = df.sort_values(["sort_key", "metal", "year"], ascending=[True, True, False])

df_sorted["include_final"] = ""
df_sorted["reason_final"] = ""
df_sorted["genes_found"] = ""

print(df_sorted["include_auto"].value_counts())
df_sorted[["database", "metal", "year", "title", "include_auto", "reason_auto"]].head(30)

high_priority_terms = [
    "gene expression", "transcriptome", "transcriptomic", "toxicogenomic",
    "rna-seq", "differential expression", "mutant", "knockout", "rnai",
    "metallothionein", "mtf-1", "ortholog", "pathway", "gene", "genes"
]

review_terms = [
    "review", "lessons from drosophila", "model for", "mechanisms"
]

low_priority_terms = [
    "lifespan", "survival", "reproductive fitness", "locomotion",
    "motor ability", "behavior", "behaviour"
]

def priority_class(text):
    txt = str(text).lower()
    high = any(t in txt for t in high_priority_terms)
    review = any(t in txt for t in review_terms)
    low = any(t in txt for t in low_priority_terms)

    if high:
        return "high_priority"
    if review:
        return "medium_priority"
    if low:
        return "low_priority"
    return "medium_priority"

# применяем только к тем, что уже yes или maybe
df_sorted["priority_auto"] = ""

mask = df_sorted["include_auto"].isin(["yes", "maybe"])
df_sorted.loc[mask, "priority_auto"] = df_sorted.loc[mask, "screen_text"].apply(priority_class)

print(df_sorted.loc[mask, "priority_auto"].value_counts())

df_sorted.loc[mask, ["database", "metal", "year", "title", "include_auto", "priority_auto"]].head(50)

# Final priority classification uses title + abstract only.
# только title + abstract, без query
df_sorted["screen_text_clean"] = (
    df_sorted["title"].fillna("").astype(str) + " " +
    df_sorted["abstract"].fillna("").astype(str)
).str.lower()

strong_terms = [
    "gene expression",
    "transcriptome",
    "transcriptomic",
    "toxicogenomic",
    "rna-seq",
    "differential expression",
    "mutant",
    "knockout",
    "rnai",
    "metallothionein",
    "mtf-1",
    "ortholog"
]

moderate_terms = [
    "expression",
    "transcript",
    "pathway",
    "oxidative stress",
    "neuronal",
    "synaptic",
    "apoptosis",
    "mitochondria"
]

review_terms = [
    "review",
    "lessons from drosophila",
    "model for",
    "mechanisms"
]

low_priority_terms = [
    "lifespan",
    "survival",
    "reproductive fitness",
    "locomotion",
    "motor ability",
    "behavior",
    "behaviour"
]

def has_any(text, words):
    return any(w in text for w in words)

def priority_class(text):
    txt = str(text).lower()
    strong = has_any(txt, strong_terms)
    moderate = has_any(txt, moderate_terms)
    review = has_any(txt, review_terms)
    low = has_any(txt, low_priority_terms)

    if strong:
        return "high_priority"
    if moderate or review:
        return "medium_priority"
    if low:
        return "low_priority"
    return "low_priority"

mask = df_sorted["include_auto"].isin(["yes", "maybe"])
df_sorted.loc[mask, "priority_auto"] = df_sorted.loc[mask, "screen_text_clean"].apply(priority_class)

print(df_sorted.loc[mask, "priority_auto"].value_counts())

df_sorted.loc[mask, ["database", "metal", "year", "title", "include_auto", "priority_auto"]].head(50)

print("Automated inclusion classes:")
print(df_sorted["include_auto"].value_counts())
print("\nPriority classes among yes/maybe records:")
print(df_sorted.loc[mask, "priority_auto"].value_counts())


In [ ]:
screening_file = OUTPUT_DIR / "screening_semiauto_strict.csv"
df_sorted.to_csv(screening_file, index=False)
print(f"Saved: {screening_file}")


## 3. High-priority corpus and text preparation

Only records assigned `high_priority` enter the automated gene-token extraction stage. Candidate extraction is deliberately permissive; false positives are removed in later filtering and manual validation stages.


In [ ]:
core_df = df_sorted[df_sorted["priority_auto"] == "high_priority"].copy()

core_df.shape

core_df["text_for_gene_search"] = (
    core_df["title"].fillna("").astype(str) + " " +
    core_df["abstract"].fillna("").astype(str)
)

print(f"High-priority records: {len(core_df):,}")


## 4. Gene-like token extraction

Two candidate sources are combined: (1) a small seed list of biologically plausible gene symbols used only to ensure capture of known notation variants, and (2) generic regular-expression patterns for gene-like tokens. These expressions generate candidates and do **not** establish gene identity or metal association.


In [ ]:
import re
import pandas as pd

seed_genes = [
    "MTF-1", "MtnA", "MtnB", "MtnC", "MtnD",
    "Sod1", "Sod2", "Cat", "GstD1", "GstE1",
    "Hsp70", "Hsp83", "p53", "reaper", "hid",
    "para", "cac", "Sh", "Syn",
    "CncC", "Keap1", "JNK", "Thor"
]

def find_seed_genes(text, gene_list):
    text = str(text)
    found = []
    for gene in gene_list:
        pattern = r"\b" + re.escape(gene) + r"\b"
        if re.search(pattern, text, flags=re.IGNORECASE):
            found.append(gene)
    return sorted(set(found))

core_df["seed_genes_found"] = core_df["text_for_gene_search"].apply(
    lambda x: find_seed_genes(x, seed_genes)
)

core_df[["title", "seed_genes_found"]].head(10)

# простые стоп-слова, чтобы убрать очевидный шум
stop_words = {
    "DNA", "RNA", "ATP", "ROS", "Cd", "Pb", "Hg",
    "Drosophila", "Genes", "Gene", "Metal", "Stress",
    "Lead", "Mercury", "Cadmium", "Review"
}

def extract_gene_like_tokens(text):
    text = str(text)

    # шаблоны под короткие gene-like названия:
    patterns = [
        r"\b[A-Z][a-z]{1,4}\d{0,2}\b",      # Sod1, Cat, Syn
        r"\b[A-Z]{2,5}-\d\b",               # MTF-1
        r"\b[a-z]{2,10}\b"                  # para, reaper, hid, notch-like lowercase genes
    ]

    hits = []
    for pat in patterns:
        hits.extend(re.findall(pat, text))

    # чистка
    cleaned = []
    for h in hits:
        if h in stop_words:
            continue
        if len(h) < 3:
            continue
        cleaned.append(h)

    return sorted(set(cleaned))

core_df["gene_like_tokens"] = core_df["text_for_gene_search"].apply(extract_gene_like_tokens)

core_df[["title", "gene_like_tokens"]].head(10)

def merge_gene_lists(row):
    merged = set(row["seed_genes_found"]) | set(row["gene_like_tokens"])
    return sorted(merged)

core_df["candidate_genes_auto"] = core_df.apply(merge_gene_lists, axis=1)


## 5. Expand candidates to one gene-like token per literature record

Candidate tokens are expanded to long format and deduplicated within each metal × title × normalized-token combination.


In [ ]:
rows = []

for _, row in core_df.iterrows():
    for gene in row["candidate_genes_auto"]:
        rows.append({
            "metal": row.get("metal", ""),
            "database_found": row.get("database", ""),
            "search_query": row.get("query", ""),
            "title": row.get("title", ""),
            "year": row.get("year", ""),
            "doi": row.get("doi", ""),
            "pmid": row.get("pmid", ""),
            "gene_symbol_raw": gene,
            "evidence_stage": "auto_extracted_candidate",
            "evidence_type": "title_abstract_text_mining",
            "validation_status": "",
            "gene_symbol_validated": "",
            "flybase_id": "",
            "known_function": "",
            "putative_role_in_neurotoxicity": "",
            "include_final": ""
        })

gene_candidates_df = pd.DataFrame(rows)
gene_candidates_df.head()
gene_candidates_df.shape

gene_candidates_df["gene_symbol_raw_norm"] = (
    gene_candidates_df["gene_symbol_raw"]
    .astype(str)
    .str.strip()
    .str.lower()
)

gene_candidates_df = gene_candidates_df.drop_duplicates(
    subset=["metal", "title", "gene_symbol_raw_norm"]
).copy()

gene_candidates_df.shape

print(f"Candidate mentions after record-level deduplication: {len(gene_candidates_df):,}")
assert len(gene_candidates_df) == 19095, "Expected 19,095 candidate mentions for the archived input corpus."


In [ ]:
candidate_file = OUTPUT_DIR / "gene_candidates_auto.csv"
gene_candidates_df.to_csv(candidate_file, index=False)
print(f"Saved: {candidate_file}")


## 6. First-stage token filtering

The original workflow then applies simple syntax and stop-word filters and retains one representative record per normalized token for the first validation stage.


In [ ]:
df = gene_candidates_df.copy()

# 1. длина >= 3
df = df[df["gene_symbol_raw"].str.len() >= 3]

# 2. убираем слова с маленькой буквы (оставляем gene-like)
df = df[df["gene_symbol_raw"].str.match(r'^[A-Za-z0-9\-]+$')]

# 3. убираем слова с пробелами
df = df[~df["gene_symbol_raw"].str.contains(" ")]

# 4. убираем полностью lowercase слова (часто шум)
df = df[~df["gene_symbol_raw"].str.islower()]

df.shape

stop_words = [
    "Biol", "Dev", "New", "Mar", "Apr", "May", "Jun",
    "Jul", "Aug", "Sep", "Oct", "Nov", "Dec",
    "Article", "Study", "Effect", "Role", "Analysis",
    "Data", "Results", "Using", "Based"
]

df = df[~df["gene_symbol_raw"].isin(stop_words)]

df.shape

df["gene_symbol_raw_norm"] = df["gene_symbol_raw"].str.lower()

df_unique = df.drop_duplicates(subset=["gene_symbol_raw_norm"]).copy()

df_unique.shape

print(f"Unique tokens after first-stage filtering: {len(df_unique):,}")


In [ ]:
validation_file = OUTPUT_DIR / "validation_1_stage.csv"
df_unique.to_csv(validation_file, index=False)
print(f"Saved: {validation_file}")


## 7. Second-stage gene-like filtering and relevance scoring

The first-stage table is normalized again, filtered using the original gene-like rules, and scored using the original relevance terms. This scoring prioritizes candidates for manual validation; it is not evidence of biological function or metal causality.


In [ ]:
df = df_unique.copy()
# Match the original CSV round-trip: missing candidate symbols are not treated as the literal token "nan".
df = df[df["gene_symbol_raw"].notna()].copy()
df = df[df["gene_symbol_raw"].astype(str).str.lower() != "none"].copy()
df["gene_symbol_raw"] = df["gene_symbol_raw"].astype(str).str.strip()

# нормализованный вариант для сравнения
df["gene_symbol_norm"] = df["gene_symbol_raw"].str.strip().str.lower()

stop_words = {
    "biol", "dev", "new", "mar", "apr", "may", "jun", "jul", "aug", "sep", "oct", "nov", "dec",
    "article", "study", "effect", "effects", "role", "analysis", "data", "results", "using",
    "stress", "metal", "metals", "heavy", "cadmium", "lead", "mercury", "drosophila",
    "gene", "genes", "pathway", "neurotoxicity", "cell", "cells", "protein", "proteins",
    "dna", "rna", "atp", "ros"
}

# 1. длина
df = df[df["gene_symbol_raw"].str.len() >= 3].copy()

# 2. только "словоподобные" символы
df = df[df["gene_symbol_raw"].str.match(r"^[A-Za-z0-9\-]+$", na=False)].copy()

# 3. убрать полностью lowercase общеязыковые слова
df = df[~df["gene_symbol_norm"].isin(stop_words)].copy()

# 4. убрать токены с одними цифрами
df = df[~df["gene_symbol_raw"].str.match(r"^\d+$", na=False)].copy()

df.shape

def gene_like_rule(symbol):
    s = str(symbol)

    patterns = [
        r"^[A-Z][a-z]{1,4}\d{0,2}$",     # Sod1, Cat, Syn
        r"^[A-Z]{2,5}-\d$",              # MTF-1
        r"^[A-Z][A-Za-z]{1,8}$",         # Notch-like
        r"^[a-z]{3,10}$",                # para, reaper, hid
        r"^[A-Za-z]{2,8}\d{1,3}$"        # GstD1, Hsp70
    ]
    return any(re.match(p, s) for p in patterns)

df["is_gene_like"] = df["gene_symbol_raw"].apply(gene_like_rule)

df = df[df["is_gene_like"]].copy()

df.shape

gene_unique = df.drop_duplicates(subset=["gene_symbol_norm"]).copy()

gene_unique.shape
gene_unique.sort_values("gene_symbol_raw").head(50)

for col in ["title", "abstract", "metal", "search_query"]:
    if col not in df.columns:
        df[col] = ""

df["context_text"] = (
    df["title"].fillna("").astype(str) + " " +
    df["abstract"].fillna("").astype(str) + " " +
    df["metal"].fillna("").astype(str) + " " +
    df["search_query"].fillna("").astype(str)
).str.lower()

strong_relevance_terms = [
    "gene expression", "transcriptome", "transcriptomic", "toxicogenomic",
    "rna-seq", "differential expression", "mutant", "knockout", "rnai",
    "metallothionein", "mtf-1"
]

moderate_relevance_terms = [
    "oxidative stress", "neuronal", "synaptic", "apoptosis",
    "mitochondria", "pathway", "calcium", "detoxification"
]

review_terms = [
    "review", "mechanisms", "model"
]

def score_relevance(text):
    txt = str(text).lower()
    score = 0

    for term in strong_relevance_terms:
        if term in txt:
            score += 3

    for term in moderate_relevance_terms:
        if term in txt:
            score += 1

    for term in review_terms:
        if term in txt:
            score += 1

    return score

df["relevance_score_auto"] = df["context_text"].apply(score_relevance)

def score_class(score):
    if score >= 4:
        return "high"
    elif score >= 2:
        return "medium"
    else:
        return "low"

df["relevance_class_auto"] = df["relevance_score_auto"].apply(score_class)

gene_summary = (
    df.groupby("gene_symbol_norm")
      .agg(
          gene_symbol_raw=("gene_symbol_raw", "first"),
          n_records=("gene_symbol_raw", "size"),
          metals_found=("metal", lambda x: ";".join(sorted(set([str(i) for i in x if pd.notna(i)])))),
          max_relevance_score=("relevance_score_auto", "max"),
          best_relevance_class=("relevance_class_auto", lambda x: sorted(set(x), key=lambda y: {"high":0, "medium":1, "low":2}[y])[0]),
          example_title=("title", "first")
      )
      .reset_index()
)

gene_summary.shape
gene_summary.sort_values(["max_relevance_score", "n_records"], ascending=[False, False]).head(50)

gene_summary["validation_status_auto"] = ""
gene_summary["gene_symbol_validated"] = ""
gene_summary["flybase_id"] = ""
gene_summary["relevance_final_manual"] = ""
gene_summary["include_final"] = ""
gene_summary["comments"] = ""

shortlist = gene_summary[
    (gene_summary["best_relevance_class"].isin(["high", "medium"]))
].copy()

shortlist.shape
shortlist.sort_values(["max_relevance_score", "n_records"], ascending=[False, False]).head(100)

print(f"Gene summary rows: {len(gene_summary):,}")
print(f"Manual-validation shortlist: {len(shortlist):,}")
assert len(shortlist) == 1053, "Expected 1,053 candidates in the archived workflow."


## 8. Export automated summary and manual-validation shortlist

The 1,053-row shortlist is the endpoint of this notebook. FlyBase identifier checking, correction of ambiguous mappings, and evidence-based reassessment are performed downstream and are not automated here.


In [ ]:
gene_summary_file = OUTPUT_DIR / "gene_summary_auto.csv"
shortlist_file = OUTPUT_DIR / "gene_shortlist_for_manual_validation.csv"

gene_summary.to_csv(gene_summary_file, index=False)
shortlist.to_csv(shortlist_file, index=False)

print(f"Saved: {gene_summary_file}")
print(f"Saved: {shortlist_file}")


## 9. Reproducibility summary

For the archived 23 April 2026 literature corpus, the expected checkpoints are:

- literature records: **507**;
- automated candidate mentions after record-level deduplication: **19,095**;
- candidates selected for subsequent manual validation: **1,053**.

The later transition from 1,053 candidates to 85 identifier-validated candidates involved manual/identifier validation and is documented separately. The evidence-based reassessment from 85 to the final 48-gene analytical set is provided in `data/Table_S1_gene_validation_85.csv`.
